# Intervals for a mean response and for a new observation

A confidence interval for a coefficient is not an interval for a prediction. At a chosen $x_0$ there are two different targets: the mean response on the line, and a single new observation that still carries its own error.


## The two standard errors

For a simple regression, the estimated mean response at $x_0$ is $\widehat y(x_0) = b_0 + b_1 x_0$. Its variance, after replacing $\sigma^2$ by $s^2$, has standard error

$$
\operatorname{se}\!\left(\widehat y(x_0)\right)
= s\sqrt{\dfrac{1}{n} + \dfrac{(x_0 - \bar x)^2}{S_{xx}}}.
$$

A new observation is that mean plus a fresh error of variance $\sigma^2$, independent of the sample. The prediction standard error is therefore larger:

$$
\operatorname{se}\!\left(y_{\mathrm{new}}\right)
= s\sqrt{1 + \dfrac{1}{n} + \dfrac{(x_0 - \bar x)^2}{S_{xx}}}.
$$

Both intervals use the $t_{n-p}$ quantile. The extra $1$ under the second square root is the new observation's own noise, and it does not shrink as $n$ grows. More data pin down the line; they do not make a new individual less variable.


## At the center, and one step outside

On the five-point line, $b_0 = 1$, $b_1 = 2$, $s^2 = 4/3$, $n = 5$, $\bar x = 3$, and $S_{xx} = 10$.

At $x_0 = 3$, the fitted value is $1 + 6 = 7$, and $(x_0-\bar x)^2 = 0$. Thus

$$
\begin{aligned}
\operatorname{se}(\widehat y) &= \sqrt{\dfrac{4}{3}}\sqrt{\dfrac{1}{5}} = \dfrac{2}{\sqrt{15}}, \\[4pt]
\operatorname{se}(y_{\mathrm{new}}) &= \sqrt{\dfrac{4}{3}}\sqrt{1 + \dfrac{1}{5}} = \sqrt{\dfrac{4}{3}\cdot\dfrac{6}{5}} = \sqrt{\dfrac{8}{5}}.
\end{aligned}
$$

At $x_0 = 6$, which is one unit beyond the last observation, the fitted value is $1 + 12 = 13$ and $(6-3)^2/S_{xx} = 9/10$. Thus

$$
\begin{aligned}
\operatorname{se}(\widehat y) &= \sqrt{\dfrac{4}{3}\cdot\dfrac{11}{10}} = \sqrt{\dfrac{22}{15}}, \\[4pt]
\operatorname{se}(y_{\mathrm{new}}) &= \sqrt{\dfrac{4}{3}\cdot\left(1 + \dfrac{1}{5} + \dfrac{9}{10}\right)} = \sqrt{\dfrac{4}{3}\cdot\dfrac{21}{10}} = \sqrt{\dfrac{14}{5}}.
\end{aligned}
$$

The mean-response interval got wider as soon as $x_0$ left $\bar x$. The prediction interval was already wide at the center, because of the extra $1$, and it got wider still.


In [1]:
# Compare the two standard errors at x=3 and at x=6.
import math
s = math.sqrt(4 / 3)

def standard_errors(x0):
    leverage_term = (x0 - 3) ** 2 / 10
    se_mean = s * math.sqrt(1 / 5 + leverage_term)
    se_new = s * math.sqrt(1 + 1 / 5 + leverage_term)
    return se_mean, se_new

at_center = standard_errors(3)
outside = standard_errors(6)
print("at 3", at_center)
print("at 6", outside)
assert abs(at_center[0] - 2 / math.sqrt(15)) < 1e-12
assert abs(at_center[1] - math.sqrt(8 / 5)) < 1e-12
assert abs(outside[0] - math.sqrt(22 / 15)) < 1e-12
assert abs(outside[1] - math.sqrt(14 / 5)) < 1e-12
assert at_center[0] < outside[0] < outside[1]
assert at_center[1] < outside[1]


at 3 (0.5163977794943222, 1.2649110640673515)
at 6 (1.2110601416389968, 1.6733200530681511)


## The intervals themselves

With $t_{3,\,0.975}$ denoting the same quantile as in the coefficient lesson, the $95\%$ intervals at $x_0 = 3$ are

$$
7 \pm t_{3,\,0.975}\cdot\dfrac{2}{\sqrt{15}}
\qquad\text{and}\qquad
7 \pm t_{3,\,0.975}\cdot\sqrt{\dfrac{8}{5}}.
$$

At $x_0 = 6$,

$$
13 \pm t_{3,\,0.975}\cdot\sqrt{\dfrac{22}{15}}
\qquad\text{and}\qquad
13 \pm t_{3,\,0.975}\cdot\sqrt{\dfrac{14}{5}}.
$$

The code multiplies these exact standard errors by the $t$ quantile. The prediction interval at the center is already several times wider than the interval for the mean response.


In [2]:
# Form the four intervals. Prediction is wider than the mean response.
import math
import scipy.stats as stats
tcrit = stats.t.ppf(0.975, 3)
mean_center = (7 - tcrit * 2 / math.sqrt(15), 7 + tcrit * 2 / math.sqrt(15))
pred_center = (7 - tcrit * math.sqrt(8 / 5), 7 + tcrit * math.sqrt(8 / 5))
print("mean response at 3", mean_center)
print("new observation at 3", pred_center)
mean_width = mean_center[1] - mean_center[0]
pred_width = pred_center[1] - pred_center[0]
print("width ratio", pred_width / mean_width)
assert pred_width > 2 * mean_width


mean response at 3 (np.float64(5.3565917945915835), np.float64(8.643408205408416))
new observation at 3 (np.float64(2.9744884576463733), np.float64(11.025511542353627))
width ratio 2.4494897427831783


## Pitfall: a long extrapolation

At $x_0 = 30$ the point prediction is still the line, $1 + 2\cdot 30 = 61$. The prediction standard error is

$$
s\sqrt{1 + \dfrac{1}{5} + \dfrac{(30-3)^2}{10}}
= \sqrt{\dfrac{4}{3}\cdot\dfrac{741}{10}}.
$$

At $x_0 = 3$ the quantity under the outer square root, after removing $s$, was $6/5$. The ratio of the two prediction standard errors is

$$
\sqrt{\dfrac{741/10}{6/5}} = \sqrt{\dfrac{247}{4}} = \dfrac{\sqrt{247}}{2} \approx 7.86.
$$

The line returns a number. The interval says the number is poorly tied to the data. Extrapolation is not a failure of the arithmetic. It is a rapid growth of $(x_0-\bar x)^2/S_{xx}$.


In [3]:
# Far from the data, the prediction standard error grows like |x0 - xbar|.
import math
s = math.sqrt(4 / 3)
se_far = s * math.sqrt(1 + 1 / 5 + (30 - 3) ** 2 / 10)
se_center = math.sqrt(8 / 5)
ratio = se_far / se_center
print("prediction at 30", 1 + 2 * 30)
print("se far", se_far)
print("ratio to the center", ratio)
assert abs(se_far - math.sqrt((4 / 3) * (741 / 10))) < 1e-12
assert abs(ratio - math.sqrt(247) / 2) < 1e-12
assert ratio > 7


prediction at 30 61
se far 9.93981891183134
ratio to the center 7.858116822750854


## Summary

- The mean response at $x_0$ has standard error $s\sqrt{1/n + (x_0-\bar x)^2/S_{xx}}$.
- A new observation adds $1$ inside that square root. More data do not remove it.
- At $x=30$ on this design, that prediction standard error is $\sqrt{247}/2$ times the prediction standard error at $x=3$.
